# Ollama on Colab, exposed over an ngrok tunnel

Runtime -> Change runtime type -> T4 GPU, before running any cell.

This notebook installs Ollama, pulls two models, serves them, and opens an
`ngrok` tunnel so `vishwakarma` (running on your local machine) can reach
this Colab GPU as the `ollama-colab` fallback provider.

**Why ngrok, not cloudflared (#71):** a `cloudflared` quick tunnel
(`trycloudflare.com`) was tried first and consistently returned `403
Forbidden` directly from Cloudflare's edge on every request -- reproduced
across 3 fresh tunnels, both QUIC and HTTP/2, from two independent networks.
ngrok's tunnel reaches the agent cleanly; the only fix needed is passing
`host_header="localhost:11434"` to `ngrok.connect()` (see the tunnel cell
below) -- without it, Ollama's own server-side Host-header check (a
DNS-rebinding protection, unrelated to `OLLAMA_ORIGINS`, which only governs
browser CORS `Origin` headers) rejects the tunnel's public hostname with a
`403` of its own. Confirmed live: `ollama.log` showed the 403 originating
from Ollama itself (`GIN ... 403 ... GET "/v1/models"`), not from ngrok.

**ngrok needs a free account + authtoken** (https://dashboard.ngrok.com/signup,
then https://dashboard.ngrok.com/get-started/your-authtoken) -- paste it into
the `NGROK_AUTHTOKEN` cell below. This is a one-time step per Google/ngrok
account, not per session.

The session is ephemeral: Colab disconnects after ~12h, on idle, or
sometimes silently drops the backend while the notebook UI still shows
"Connected" -- if `ollama`/model pulls suddenly report missing files, the
runtime was actually reset and every cell from the top needs a re-run
(installs and model pulls included). A fresh session also gets a new tunnel
URL. Copy the last cell's output into `OLLAMA_COLAB_BASE_URL` in your local
`.env` after every restart -- see `models.yaml`'s `ollama-colab` block and
the README's "Colab-hosted Ollama fallback" section for why it's an env var
and not a line in `models.yaml`.

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
os.environ["OLLAMA_ORIGINS"] = "*"
!nohup ollama serve > ollama.log 2>&1 &

Pulls both models this notebook is verified against. If a pull fails, the
tag doesn't exist in Ollama's library -- swap it for another real,
locally-run tag (avoid any `:cloud`-suffixed tag; those run on Ollama's own
cloud infra and return `401 Unauthorized` without a separate `ollama.com`
sign-in, not on this GPU), and make the matching one-line edit to the
`ollama-colab` candidates in `models.yaml`.

In [ ]:
import time
time.sleep(5)
!ollama pull qwen2.5-coder:7b
!ollama pull deepseek-coder-v2:16b

Paste your ngrok authtoken below (from
https://dashboard.ngrok.com/get-started/your-authtoken).

In [ ]:
NGROK_AUTHTOKEN = "paste-your-authtoken-here"

!pip -q install pyngrok

`host_header="localhost:11434"` is required -- see the note at the top of
this notebook for why Ollama 403s the tunnel's own public hostname without it.

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(11434, "http", host_header="localhost:11434")
print(tunnel.public_url)

Sanity check -- confirms the tunnel actually reaches Ollama (not just ngrok's
own edge) before you copy the URL into `.env`.

In [ ]:
import time
time.sleep(2)
!curl -s -o /dev/null -w "tunnel reachability: %{http_code}\n" {tunnel.public_url}/v1/models

Public URL -- copy this into `.env` as `OLLAMA_COLAB_BASE_URL=<url>/v1` (a
200 above confirms it's actually live, not just printed):

In [ ]:
print(f"{tunnel.public_url}/v1")